# Insurance Reducto OracleDB Demo Notebook

This notebook demonstrates the insurance version of the Reducto + Oracle flow:

1. Load `.env`.
2. Connect to Oracle and ensure the schema exists.
3. Inspect documents tagged with `filing_type = 'INSURANCE'`.
4. Preview extracted tables for coverage schedules, limits, deductibles, or claim summaries.
5. Retrieve chunk evidence without requiring a live embedding call.
6. Optionally ingest an insurance document URL through Reducto.

## Platform Context

![Executive thesis: Reducto AI + Oracle Database](assets/reducto_oracle_executive_thesis.svg)

![Capability map: Oracle Database approach vs standalone vector platforms](assets/reducto_oracle_platform_capability_map.svg)

![Operating model and risk controls](assets/reducto_oracle_operating_model_risk_controls.svg)


In [2]:
import os
import sys
from pathlib import Path
from dataclasses import replace


def find_sdk_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "src" / "reducto" / "lib" / "oracledb").exists():
            return path
    raise RuntimeError("Could not locate the reducto-python-sdk repository root")


ROOT = find_sdk_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


def load_env(path=ROOT / "examples" / "oracledb" / ".env"):
    if not path.exists():
        return
    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        if line.startswith("export "):
            line = line[len("export ") :].strip()
        key, value = line.split("=", 1)
        os.environ[key.strip()] = value.strip().strip("'\"")


def optional_int(name):
    value = os.getenv(name)
    return int(value) if value else None


load_env()

DOMAIN_LABEL = "INSURANCE"
DOMAIN_ENTITY = os.getenv("INSURANCE_ENTITY", "DEMO_INSURANCE")
DOMAIN_YEAR = optional_int("INSURANCE_YEAR")
QUESTION = os.getenv(
    "INSURANCE_DEMO_QUESTION",
    "What coverage, exclusions, deductibles, limits, or claim decision are documented?",
)

print("Oracle user:", os.getenv("ORACLE_USER"))
print("Oracle DSN:", os.getenv("ORACLE_DSN"))
print("Reducto key set:", bool(os.getenv("REDUCTO_API_KEY")))
print("Cohere key set:", bool(os.getenv("CO_API_KEY")))
print("Domain label:", DOMAIN_LABEL)

Oracle user: REDUCTO_RAG_APP
Oracle DSN: localhost:1521/FREEPDB1
Reducto key set: True
Cohere key set: True
Domain label: INSURANCE


## Connect to Oracle and inspect the schema

In [3]:
from reducto.lib.oracledb.config import connect_oracle, vector_dimensions_from_env
from reducto.lib.oracledb.oracle import OracleSchemaManager

connection = connect_oracle()
OracleSchemaManager(connection).create_schema(vector_dimensions=vector_dimensions_from_env())
print("Oracle schema is ready")

for table in ["DOCUMENTS", "DOCUMENT_CHUNKS", "PARSED_TABLES", "FINANCIAL_FACTS"]:
    with connection.cursor() as cursor:
        cursor.execute(f"select count(*) from {table}")
        print(table, cursor.fetchone()[0])

Oracle schema is ready
DOCUMENTS 2
DOCUMENT_CHUNKS 21
PARSED_TABLES 47
FINANCIAL_FACTS 1222


## Ingest an insurance URL

Set `RUN_INSURANCE_INGEST=1` and `INSURANCE_SOURCE_URI=<url>` before running this cell. The notebook keeps parsed tables but clears `financial_facts`, because those facts are tuned for SEC-style financial statements.

In [ ]:
RUN_INGEST = os.getenv("RUN_INSURANCE_INGEST") == "1"
SOURCE_URI = os.getenv("INSURANCE_SOURCE_URI", "")

if not RUN_INGEST:
    print("Skipping ingest. Set RUN_INSURANCE_INGEST=1 and INSURANCE_SOURCE_URI to parse a document.")
elif not SOURCE_URI:
    raise RuntimeError("INSURANCE_SOURCE_URI must be set when RUN_INSURANCE_INGEST=1")
else:
    from reducto.lib.oracledb.models import DocumentMetadata
    from reducto.lib.oracledb.oracle import OracleDocumentRepository
    from reducto.lib.oracledb.embeddings import embedding_provider_from_env
    from reducto.lib.oracledb.reducto_client import ReductoDocumentParser

    parse_result = ReductoDocumentParser().parse_url(SOURCE_URI)
    parse_result = replace(parse_result, financial_facts=[])
    metadata = DocumentMetadata(
        source_uri=SOURCE_URI,
        source_kind="url",
        company=DOMAIN_ENTITY,
        fiscal_year=DOMAIN_YEAR,
        filing_type=DOMAIN_LABEL,
        title=os.getenv("INSURANCE_TITLE", "Insurance Reducto demo document"),
    )
    document_id = OracleDocumentRepository(connection).store_parse_result(
        metadata,
        parse_result,
        embedding_provider_from_env(input_type="search_document"),
    )
    print("Stored insurance document", document_id)
    print("Chunks:", len(parse_result.chunks), "Tables:", len(parse_result.tables))

## List stored insurance documents

In [6]:
def fetch_domain_documents(limit=10):
    with connection.cursor() as cursor:
        cursor.execute(
            f"""
            select document_id, company, fiscal_year, filing_type, title, source_uri
            from documents
            where upper(filing_type) = upper(:domain_label)
            order by created_at desc
            fetch first {max(1, int(limit))} rows only
            """,
            domain_label=DOMAIN_LABEL,
        )
        return cursor.fetchall()


domain_docs = fetch_domain_documents()
if not domain_docs:
    print("No INSURANCE documents found yet. Use the optional ingest cell below to add one.")
else:
    for row in domain_docs:
        print(row)

(41, 'CMS', 2024, 'INSURANCE', 'CMS Summary of Benefits and Coverage Template', 'https://www.cms.gov/cciio/resources/regulations-and-guidance/downloads/sbc-template.pdf')


## Preview extracted insurance tables

For insurance documents, tables are useful for declarations pages, coverage schedules, limits, deductibles, endorsements, and claim payment summaries. The financial-fact promotion table is intentionally not used here.

In [7]:
from reducto.lib.oracledb.utils import read_lob

if not domain_docs:
    print("Skipping table preview because no INSURANCE documents are stored yet.")
else:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            select d.document_id, d.title, t.table_index, t.page_number, t.raw_content
            from parsed_tables t
            join documents d on d.document_id = t.document_id
            where upper(d.filing_type) = upper(:domain_label)
            order by d.created_at desc, t.table_index
            fetch first 5 rows only
            """,
            domain_label=DOMAIN_LABEL,
        )
        table_rows = cursor.fetchall()

    if not table_rows:
        print("No parsed insurance tables found. Chunk search can still work without tables.")
    for document_id, title, table_index, page_number, raw_content in table_rows:
        text = str(read_lob(raw_content) or "")
        print(f"Document {document_id} | table {table_index} | page {page_number} | {title}")
        print(text[:700].replace("\n", " "))
        print("---")

Document 41 | table 0 | page 1 | CMS Summary of Benefits and Coverage Template
Important Questions,Answers ,Why This Matters: "What is the overall deductible? ",$, "Are there services covered before you meet your deductible?",, "Are there other deductibles for specific services? ",$, "What is the out-of-pocket limit for this plan?",$, "What is not included in the out-of-pocket limit?",, "Will you pay less if you use a network provider?",, "Do you need a referral to see a specialist?",, 
---
Document 41 | table 1 | page 2 | CMS Summary of Benefits and Coverage Template
"""Common Medical Event"",Services You May Need,What You Will Pay,What You Will Pay,""Limitations, Exceptions, &amp; Other Important Information"" ""Common Medical Event"",Services You May Need,""Network Provider (You will pay the least)"",""Out-of-Network Provider (You will pay the most)"",""Limitations, Exceptions, &amp; Other Important Information"" ""If you visit a health care provider's office or clinic"",""Primary c

## Retrieve insurance chunk evidence

This cell uses a lightweight SQL keyword search so the notebook can run without a live embedding request. Set `RUN_LIVE_EMBED_SEARCH=1` in your environment to run semantic search in the next cell.

In [8]:
from reducto.lib.oracledb.qa import format_answer, answer_from_search_results
from reducto.lib.oracledb.models import SearchResult

SEARCH_TERMS = ["coverage", "exclusion", "deductible", "premium", "claim", "limit"]


def lexical_chunk_search(terms, limit=5):
    clauses = []
    params = {"domain_label": DOMAIN_LABEL}
    for index, term in enumerate(terms):
        key = f"term_{index}"
        clauses.append(f"lower(c.content) like :{key}")
        params[key] = f"%{term.lower()}%"
    term_sql = " or ".join(clauses)
    with connection.cursor() as cursor:
        cursor.execute(
            f"""
            select d.document_id, c.chunk_id, c.content, d.company, d.fiscal_year,
                   d.filing_type, c.page_start, c.page_end, d.source_uri
            from document_chunks c
            join documents d on d.document_id = c.document_id
            where upper(d.filing_type) = upper(:domain_label)
              and ({term_sql})
            order by c.created_at desc
            fetch first {max(1, int(limit))} rows only
            """,
            params,
        )
        rows = cursor.fetchall()

    results = []
    for row in rows:
        content = str(read_lob(row[2]) or "")
        score = sum(1 for term in terms if term.lower() in content.lower())
        results.append(
            SearchResult(
                document_id=int(row[0]),
                chunk_id=int(row[1]),
                score=float(score),
                content=content,
                company=row[3],
                fiscal_year=int(row[4]) if row[4] is not None else None,
                filing_type=row[5],
                page_start=int(row[6]) if row[6] is not None else None,
                page_end=int(row[7]) if row[7] is not None else None,
                source_uri=str(row[8] or ""),
            )
        )
    return results


if not domain_docs:
    print("Skipping evidence retrieval because no INSURANCE documents are stored yet.")
else:
    results = lexical_chunk_search(SEARCH_TERMS, limit=5)
    if not results:
        print("No keyword matches found for:", SEARCH_TERMS)
    else:
        print(format_answer(answer_from_search_results(QUESTION, results, evidence_limit=3)))

Question
  What deductibles, coverage limits, exclusions, and appeal rights are described?

Answer
  Please note these coverage examples are based on self-only coverage.## Peg is Having a Baby (9 months of in-network pre-natal care and a hospital delivery) "The plan's overall deductible,$ Specialist copayment,$ Hospital (facility) coinsurance,% Other coinsurance ,%" "Total Example Cost,$ ""In this example, Peg would pay:"",""In this example, Peg would pay:"" ""Cost Sharing Deductibles $ "",""Cost Sharing Deductibles $ "" , Copayments ,$ , ""Coinsurance $ What isn't covered Limits or exclusio...

Evidence
  1. page 5
     Please note these coverage examples are based on self-only coverage.## Peg is Having a Baby (9 months of in-network pre-natal care and a hospital delivery) "The plan's overall deductible,$ Specialist copayment,$ Hospital (facility) coinsurance,% Other coinsurance ,%" "Total Example Cost,$ ""In this example, Peg would pay:"",""In this example, Peg would pay:"" ""Cost Sh

## Optional semantic search

This calls the configured embedding provider, so it is opt-in.

In [ ]:
RUN_LIVE_EMBED_SEARCH = os.getenv("RUN_LIVE_EMBED_SEARCH") == "1"

if not RUN_LIVE_EMBED_SEARCH:
    print("Skipping live semantic search. Set RUN_LIVE_EMBED_SEARCH=1 to enable it.")
elif not domain_docs:
    print("Skipping semantic search because no INSURANCE documents are stored yet.")
else:
    from reducto.lib.oracledb.models import SearchFilters
    from reducto.lib.oracledb.retrieval import OracleHybridRetriever
    from reducto.lib.oracledb.embeddings import embedding_provider_from_env

    retriever = OracleHybridRetriever(
        connection,
        embedding_provider_from_env(input_type="search_query"),
    )
    semantic_results = retriever.semantic_search(
        QUESTION,
        filters=SearchFilters(filing_type=DOMAIN_LABEL),
        limit=5,
    )
    print(format_answer(answer_from_search_results(QUESTION, semantic_results, evidence_limit=3)))

Question
  What deductibles, coverage limits, exclusions, and appeal rights are described?

Answer
  e Cost In this example, Joe would pay: Cost Sharing $## Mia's Simple Fracture (in-network emergency room visit and follow up care) "■The plan's overall deductible,A ■Specialist copayment,sa $ Hospital (facility) coinsurance,olo Other coinsurance,olo %" "Total Example Cost,$ ""In this example, Mia would pay:"",""In this example, Mia would pay:"" ""Cost Sharing Deductibles $ "",""Cost Sharing Deductibles $ "" , Copayments ,$ , ""Coinsurance $ What isn't covered Limits or exclusions $ "",""Coins...

Evidence
  1. page 5
     e Cost In this example, Joe would pay: Cost Sharing $## Mia's Simple Fracture (in-network emergency room visit and follow up care) "■The plan's overall deductible,A ■Specialist copayment,sa $ Hospital (facility) coinsurance,olo Other coinsurance,olo %" "Total Example Cost,$ ""In this example, Mia would pay:"",""In this example, Mia would pay:"" ""Cost Sharing Deductibl

In [8]:
connection.close()
print("Connection closed")

Connection closed
